# 时延压力与预算下的多智能体团队

我们的多智能体团队由一个主智能体组成，它能够派生子智能体来并行完成任务的一部分。如果放任不管，团队追求的是周全而不是速度：任何智能体的上下文里都没有告诉提问者已经等了多久、或者他愿意等多久。在本 cookbook 中，我们将激发 Claude 的时间意识，把任务完成得更快。

**读完本 cookbook 后，你将能够：**

- 让团队中每个智能体看到同一个运行中的时钟，同时不破坏提示词缓存
- 用一句时延压力让团队干活更快，或者对照一条
  时延预算自行配速
- 从一次运行自身的日志中检查每个智能体是否都看到了时钟

我们首先让 Claude 意识到已经过去了多少时间，只用 Messages API：

1. **一个共享的团队时钟**，在任务开始时启动。
2. **每个智能体在每一轮之前都看到时钟。** 我们每次为*任何*智能体调用 API 时，
   都在最新的用户消息末尾追加一行简短内容，`[elapsed 252s]`（第一轮是任务提示词，之后是工具结果）。在任务开始三分钟后才被派生的子智能体，第一轮看到的就是 `[elapsed 182s]`，而不是 `0s`，因为时钟属于团队，不属于某个智能体。

然后我们通过两种方式之一给 Claude 一种紧迫感：

1. **时延压力：** 在任务提示词末尾、以及每个子智能体任务说明的末尾各加一句话，说明时间很重要。
   这句话是与默认提示词之间唯一的差别。
2. **时延预算：** 增强版时钟在已用时之外还带上一条预算，
   例如 `[elapsed 252s / 600s]`。不附带任何提示词文本：第二个数字就是关于预算告知任何智能体的全部内容。

这就是全部机制，它给出三个可供比较的分支：完全没有时钟、带压力的时钟，以及带预算的时钟。

> **关于模型的说明。** 时延激励是一种新出现的引导 Claude 的方式。本笔记本
> 默认使用 `claude-fable-5-1`。该方法尚未在 Claude Opus 5 及更早的
> 模型上充分测试，因此如果你修改 `COOKBOOK_MODEL`，效果可能有所不同。

这里没有内置任务。工具、提示词和问题由你来提供；结尾一个简短的冒烟测试会展示时钟到达了每个智能体。

编排部分（消息中枢、消息工具、派生工具、智能体循环）遵循 [异步多智能体编排](async_multi_agent_orchestration.ipynb) cookbook；如果这种结构对你陌生，请从那里开始。

## 前置条件

- Python 3.11 或更高版本。
- 必备知识：能顺利阅读 `async`/`await` Python（任务、事件、超时），并
  熟悉 Messages API 的工具使用循环（一轮模型输出、它的工具使用、你送回的工具结果）。[异步多智能体编排](async_multi_agent_orchestration.ipynb) cookbook 介绍了本 cookbook 所依赖的团队机制。
- 在 `ANTHROPIC_API_KEY`（或 `.env` 文件）中有一个 Claude API 密钥。
- 一些 API 额度。每次运行是一个主智能体加最多四个子智能体，每个都会发起多次调用。我们把
  调用次数限制为每个子智能体十五次、主智能体四十次。从头到尾运行本笔记本是一次简短的团队运行，使用 `claude-fable-5-1`、思考强度 `xhigh`，大约一分钟。
- 速率限制要有余量。最多五个智能体会同时调用 API，而在较低的套餐层级上，
  SDK 的自动重试会表现为额外的墙钟时间，而不是错误。

In [1]:
%%capture
%pip install -qU anthropic python-dotenv

In [2]:
import asyncio
import itertools
import os
import re
import time
from collections import Counter, defaultdict

import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY from the environment

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-fable-5-1")  # every agent; any current model
EFFORT = os.environ.get("COOKBOOK_EFFORT", "xhigh")  # low | medium | high | xhigh | max
USER_ID = os.environ.get("COOKBOOK_USER_ID")  # optional: sent as metadata.user_id if set
MAX_HELPERS = 4  # cap on concurrently running helpers (a cost guard, not part of the mechanism)
PROGRESS_EVERY = 30  # seconds between the progress lines a run prints; None prints none

## 团队时钟

一只秒表加一个单行渲染器。每个任务只有**一个**时钟对象，团队中每个智能体都持有它的引用。正因如此，很晚才被派生的子智能体看到的是团队的已用时间，而不是自己的。可选的预算挂在同一个对象上，所以它同样属于团队：在 600 秒预算下、任务开始 182 秒后才被派生的子智能体看到的是 `[elapsed 182s / 600s]`，而不是属于它自己的全新 600 秒。

这一行只在一个地方渲染，即 `clock_line`，以整秒为单位，这样带上预算后两个数字单位一致，剩下的只是一次减法。方括号只是为了把这一行与它前面的提示词或工具结果区分开。

In [3]:
def clock_line(elapsed: float, budget: float | None = None) -> str:
    """The line the agents see: '[elapsed 192s]', or '[elapsed 192s / 600s]' with a budget."""
    line = f"elapsed {round(elapsed)}s"
    if budget is not None:
        line += f" / {round(budget)}s"
    return f"[{line}]"


class TeamClock:
    def __init__(self, budget: float | None = None):
        self.start = time.monotonic()
        self.budget = budget  # in seconds; shown on the line when set, never enforced

    @property
    def elapsed(self) -> float:
        return time.monotonic() - self.start

    def note(self) -> str:
        return clock_line(self.elapsed, self.budget)


def stamp_elapsed(messages: list[dict], clock: TeamClock) -> str:
    """Append the clock line to the end of the newest user message, and return the line.

    After the task prompt it is a text block of its own; after tool results it extends the last
    result's text. Earlier lines stay in the history, so an agent can see how long each of its
    own steps took.
    """
    line = clock.note()
    last = messages[-1]
    content = last["content"]
    if isinstance(content, str):  # the task prompt: add the line as its own text block
        last["content"] = [{"type": "text", "text": content}, {"type": "text", "text": line}]
    elif content and content[-1].get("type") == "tool_result":  # tool results: extend the last
        content[-1]["content"] += f"\n\n{line}"
    else:
        content.append({"type": "text", "text": line})
    return line


print(clock_line(192), "   with a 600-second budget:", clock_line(192, 600))

demo_clock = TeamClock()
demo_clock.start -= 192  # pretend 192 seconds have passed
demo_messages = [{"role": "user", "content": "Find the best settings for our process."}]
stamp_elapsed(demo_messages, demo_clock)
demo_messages  # this is exactly what the model receives on a first turn

[elapsed 192s]    with a 600-second budget: [elapsed 192s / 600s]


[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Find the best settings for our process.'},
   {'type': 'text', 'text': '[elapsed 192s]'}]}]

## 中枢与协调工具

**协调工具**就是编排 cookbook 中的那些客户端侧工具：每个智能体都可以通过一个内存中的中枢 `send_message` / `wait_for_message`，主智能体还可以 `create_subagents`。子智能体完成后，它的最终答案会自动送达主智能体。中枢还保存本次运行中每个智能体的工具使用次数以及日志。

In [4]:
def tool(name: str, description: str, properties: dict, required: list[str]) -> dict:
    schema = {"type": "object", "properties": properties, "required": required}
    return {"name": name, "description": description, "input_schema": schema}


SEND_MESSAGE = tool(
    "send_message",
    "Send a message to one or more other agents on your team. It is appended to their next "
    "tool result.",
    {
        "recipient_ids": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "content": {"type": "string"},
    },
    ["recipient_ids", "content"],
)
WAIT_FOR_MESSAGE = tool(
    "wait_for_message",
    "Block until another agent messages you or a helper finishes. Messages also arrive appended "
    "to the result of any other tool call.",
    {},
    [],
)
CREATE_SUBAGENTS = tool(
    "create_subagents",
    "Spawn helper agents that work in parallel with you. Returns immediately with their ids. "
    "Each helper gets base_instruction plus its own entry of per_helper_instructions, has the "
    "same task tools as you, and reports its findings back to you when it finishes.",
    {
        "base_instruction": {"type": "string"},
        "per_helper_instructions": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": MAX_HELPERS,
        },
    },
    ["base_instruction", "per_helper_instructions"],
)


class Hub:
    """One run's shared state: an inbox and an event per agent, plus the run's counts and log."""

    def __init__(self):
        self.inbox: dict[str, list[dict]] = defaultdict(list)
        self.event: dict[str, asyncio.Event] = defaultdict(asyncio.Event)
        self.status: dict[str, str] = {}
        self.usage: dict[str, Counter] = defaultdict(Counter)  # per-agent task-tool call counts
        self.log: list[dict] = []  # every agent's progress entries; run_team returns them
        self._ids = itertools.count(1)

    def register(self, name: str) -> str:
        self.status[name] = "active"
        return name

    def new_helper_name(self) -> str:
        return self.register(f"helper{next(self._ids)}")

    def post(self, sender: str, recipients: list[str], content: str) -> list[str]:
        delivered = []
        for rid in recipients:
            if self.status.get(rid) in ("active", "waiting"):  # a finished agent reads nothing
                self.inbox[rid].append({"from": sender, "content": content})
                self.event[rid].set()
                delivered.append(rid)
        return delivered

    def drain(self, name: str) -> str:
        msgs, self.inbox[name] = self.inbox[name], []
        self.event[name] = asyncio.Event()
        if not msgs:
            return ""
        body = "\n".join(
            f'<agent-message from="{m["from"]}">\n{m["content"]}\n</agent-message>' for m in msgs
        )
        return f"\n\n[Messages received while you were working:]\n{body}"

## 你的任务工具

这是需要你自己填写的部分。`TASK_TOOLS` 是每个智能体都会拿到的工具定义列表（用同一个小型 `tool` 辅助函数构建），而 `make_task_handlers(hub)` 返回执行它们的函数，每个工具名对应一个 `async fn(agent, input)`，每次运行时重新创建。两者初始为空，所以开箱即用时团队依靠模型自身的知识工作。带注释的模板展示了单个工具的形状。

> **说明：客户端侧工具能让时钟保持新鲜。** 运行框架只能在 API 调用之间添加时钟行。当一个响应
> 正在服务端采样时，包括它在那里运行的任何服务端工具，什么都无法
> 注入，所以智能体只能依据它看到的最后一条时钟行工作，直到该调用返回。每一次
> 客户端侧工具使用都会把控制权交回运行框架，而下一次请求会带上更新后的时钟行。
> 智能体的工作越多经由客户端侧工具完成，它的时钟就越保持最新。

In [5]:
TASK_TOOLS: list[dict] = []  # your tool definitions: every agent on the team gets them


def make_task_handlers(hub: Hub) -> dict:
    """The functions behind TASK_TOOLS for one run: {tool name: async fn(agent, input) -> str}."""
    return {}


# A template for one client-side tool. `agent` is the calling agent's name ("lead", "helper2", ...).
#
# TASK_TOOLS = [
#     tool(
#         "lookup_record",
#         "Look up one record by its key in the records system. Takes a few seconds.",
#         {"key": {"type": "string"}},
#         ["key"],
#     )
# ]
#
#
# def make_task_handlers(hub: Hub) -> dict:
#     async def lookup_record(agent: str, args: dict) -> str:
#         return await records.get(args["key"])  # your own code goes here
#
#     return {"lookup_record": lookup_record}

## 智能体循环，以及时钟放在哪里

一个协程运行任意智能体，无论主智能体还是子智能体。它是一个普通的工具使用循环，只有两处增补，代码中都有标注：

- **在每次 API 调用之前**，`stamp_elapsed` 把团队时钟行追加到最新的
  用户消息末尾。第一轮时该消息就是任务提示词，所以智能体在做任何事之前就知道已用时间。之后这一行会放在最后一个工具结果的末尾，与其他智能体发来的消息相邻。因为打戳发生在发送时刻，等待子智能体或等待慢工具所花的时间也会被计入。
- **当子智能体完成时**，它的最终文本被投递到主智能体的收件箱。在此之前，它会读取工作期间收到的任何
  消息，因此主智能体发来的改向不会丢失。

两个 API 细节。响应 `content`（可能包含思考）按原样写入历史；只有客户端侧的 `tool_use` 块需要我们提供 `tool_result`。而顶层的 `cache_control` 开启了[自动提示词缓存](https://platform.claude.com/docs/en/build-with-claude/prompt-caching)，这对历史被工具结果填满的智能体很重要；因为时钟行追加在末尾，它之前的内容在下一次调用时通常仍然命中缓存。

一个智能体在同一轮中的多次工具使用会并发执行。循环还会把每个智能体被展示的内容写入本次运行的日志（`[task]`、`[clock]` 和 `[report]` 条目），最后一节会把它读回来。

In [6]:
TASK_TOOL_NAMES = {t["name"] for t in TASK_TOOLS}


def log(hub: Hub, clock: TeamClock, name: str, text) -> None:
    """Record one progress entry on the run's hub. Nothing is printed."""
    text = " ".join(str(text).split())
    hub.log.append({"seconds": round(clock.elapsed), "agent": name, "text": text})


async def call_tool(hub: Hub, clock: TeamClock, name: str, block, dispatch: dict) -> dict:
    """Run one of an agent's tool calls and return the tool_result block for it."""
    if block.name == "send_message":
        ids = block.input["recipient_ids"]
        sent = hub.post(name, ids, block.input["content"])
        missed = [r for r in ids if r not in sent]
        out = f"delivered to {sent or 'no one'}" + (f"; not running: {missed}" if missed else "")
    elif block.name == "wait_for_message":
        hub.status[name] = "waiting"
        try:
            await asyncio.wait_for(hub.event[name].wait(), timeout=90)
            out = "new messages below"
        except TimeoutError:
            out = "no messages yet (waited 90s)"
        hub.status[name] = "active"
    elif block.name in dispatch:
        if block.name in TASK_TOOL_NAMES:
            hub.usage[name][block.name] += 1
        out = await dispatch[block.name](name, block.input)
    else:
        out = f"error: unknown tool {block.name}"
    log(hub, clock, name, f"-> {block.name}({block.input}) = {out}")
    return {"type": "tool_result", "tool_use_id": block.id, "content": out}


async def run_agent(
    hub: Hub,
    clock: TeamClock,
    name: str,
    *,
    system: str,
    task: str,
    tools: list[dict],
    dispatch: dict,
    report_to: str | None = None,
    show_clock: bool = True,
    max_turns: int = 40,
) -> str:
    """Run one agent to completion and return its final text."""
    messages = [{"role": "user", "content": task}]
    log(hub, clock, name, f"[task] {task}")  # what this agent was asked, kept for the run's log
    final = f"[{name} stopped after max_turns={max_turns}]"
    try:
        for _ in range(max_turns):
            if show_clock:
                line = stamp_elapsed(messages, clock)  # <- addition 1: the clock, before every call
                log(hub, clock, name, f"[clock] {line}")
            resp = await client.messages.create(
                model=MODEL,
                max_tokens=16000,  # the SDK rejects large non-streaming requests; stream for more
                system=system,
                tools=tools,
                messages=messages,
                cache_control={"type": "ephemeral"},  # automatic prompt caching
                output_config={"effort": EFFORT},
                **({"metadata": {"user_id": USER_ID}} if USER_ID else {}),
            )
            messages.append({"role": "assistant", "content": resp.content})
            texts = [b.text for b in resp.content if b.type == "text" and b.text.strip()]
            for text in texts:
                log(hub, clock, name, text)

            calls = [b for b in resp.content if b.type == "tool_use"]
            if resp.stop_reason != "tool_use" or not calls:  # end_turn (or max_tokens): finished
                # On a normal end of turn, a helper first reads whatever reached it while it worked,
                # so a redirect or a follow-up from the lead is handled rather than lost.
                mail = hub.drain(name) if report_to and resp.stop_reason == "end_turn" else ""
                if mail:
                    messages.append({"role": "user", "content": mail.lstrip()})
                    continue
                final = texts[-1].strip() if texts else f"[{name} ended with {resp.stop_reason}]"
                if resp.stop_reason == "max_tokens":  # cut off: say so, or it reads as a fast run
                    final = f"[{name} hit max_tokens; answer truncated] {final}"
                break

            # one turn's tool calls run concurrently
            results = await asyncio.gather(
                *(call_tool(hub, clock, name, b, dispatch) for b in calls)
            )
            results[-1]["content"] += hub.drain(name)  # the inbox rides on the last result
            messages.append({"role": "user", "content": list(results)})
        hub.status[name] = "done"
    except asyncio.CancelledError:
        hub.status[name] = "cancelled"
        raise
    except Exception as e:  # a crashed helper should not take the team down
        hub.status[name] = "crashed"
        final = f"[{name} crashed: {e!r}]"
        log(hub, clock, name, final)
        if report_to is None:
            raise
    if report_to:  # <- addition 2: a finished helper reports to the lead automatically
        hub.post(name, [report_to], f"(final report)\n{final}")
        log(hub, clock, name, f"[report] delivered to {report_to}")
    return final

## 提示词

- `TIME_MATTERS` 会被追加一次到主智能体的任务中，**并且**追加到每个子智能体的任务说明中。它是
  默认提示词与带压力提示词之间唯一的差别。我们试验过许多种措辞，认为这一句在时延与质量之间提供了可接受的平衡。
- 预算**不增加任何提示词文本**。设置了 `latency_budget_seconds` 后，时钟行会多出一个
  第二个数字，`[elapsed 252s / 600s]`，这就是关于预算告知任何智能体的全部内容。
- 系统提示词对时间只字未提。

`run_team(question)` 运行不带时钟的默认团队。`latency_pressure=True` 加上时钟和那句话；`latency_budget_seconds=N` 加上带预算的时钟。它们是彼此独立的分支，因此同时要求两者会报错。一次运行每 `PROGRESS_EVERY` 秒打印一行进度，并返回答案、墙钟秒数、每个智能体的工具使用次数以及完整日志。

In [7]:
TIME_MATTERS = (
    "\n\nTime matters here: do not spend time that can be avoided, and the earlier a correct "
    "result is obtained, the better."
)

LEAD_SYSTEM = (
    "You are the lead of a small team. You can work on the task yourself, and you can spawn up to "
    f"{MAX_HELPERS} helper agents with create_subagents to work on parts of it in parallel; each "
    "helper has the same task tools as you and reports back to you when it finishes (use "
    "wait_for_message to collect reports, send_message to redirect a helper). Decompose the task, "
    "delegate the independent parts, check what comes back, and finish with a short final answer "
    "that states the answer on its first line."
)
HELPER_SYSTEM = (
    "You are {name}, a helper on a team led by the agent 'lead'. Do exactly what your brief asks. "
    "When you are done, reply with your findings: the answer first, then the key evidence. That "
    "final reply is delivered to the lead automatically. If the lead messages you a follow-up, "
    "handle it the same way."
)


def total_calls(hub: Hub) -> int:
    return sum(sum(counts.values()) for counts in hub.usage.values())


async def run_team(
    question: str,
    *,
    latency_pressure: bool = False,
    latency_budget_seconds: float | None = None,
) -> dict:
    """Run one of three arms, never a mix.

    no clock (the default): the stock prompts, and no clock line.
    pressure (latency_pressure=True): the time-matters sentence, and the clock line.
    budget (latency_budget_seconds=N): the clock line with the budget on it, and no sentence.
    """
    budget = latency_budget_seconds
    if budget is not None and budget < 1:
        raise ValueError("latency_budget_seconds must be at least 1 (it is in seconds), or None")
    if latency_pressure and budget is not None:
        raise ValueError("pressure and a budget are separate arms: pass one or the other")
    hub, clock = Hub(), TeamClock(budget)  # one clock, and one budget, for the whole team
    hub.register("lead")
    helpers: dict[str, asyncio.Task] = {}
    show_clock = latency_pressure or budget is not None
    # Pressure adds one sentence to each prompt. A budget adds no words: only the clock line
    # changes, and TeamClock already carries the budget.
    time_matters = TIME_MATTERS if latency_pressure else ""
    task_handlers = make_task_handlers(hub)

    async def create_subagents(agent: str, args: dict) -> str:
        briefs = args["per_helper_instructions"]
        free = MAX_HELPERS - sum(not t.done() for t in helpers.values())
        spawned = []
        for brief in briefs[: max(free, 0)]:
            h = hub.new_helper_name()
            coro = run_agent(
                hub,
                clock,  # the TEAM clock: a helper's first line shows team elapsed, not 0s
                h,
                system=HELPER_SYSTEM.format(name=h),
                task=f"{args['base_instruction']}\n\n{brief}".strip() + time_matters,
                tools=[*TASK_TOOLS, SEND_MESSAGE, WAIT_FOR_MESSAGE],
                dispatch=task_handlers,
                report_to="lead",
                show_clock=show_clock,
                max_turns=15,
            )
            helpers[h] = asyncio.create_task(coro)
            spawned.append(h)
        held = len(briefs) - len(spawned)
        note = f" ({held} not spawned: at most {MAX_HELPERS} helpers at once)" if held else ""
        return f"spawned: {spawned}{note}"

    async def report_progress() -> None:
        while PROGRESS_EVERY:  # None or 0 turns the progress lines off
            await asyncio.sleep(PROGRESS_EVERY)
            running = sum(s in ("active", "waiting") for s in hub.status.values())
            print(
                f"[{round(clock.elapsed):>5}s] {running} agent(s) running, "
                f"{total_calls(hub)} tool calls"
            )

    progress = asyncio.create_task(report_progress())
    try:
        answer = await run_agent(
            hub,
            clock,
            "lead",
            system=LEAD_SYSTEM,
            task=question + time_matters,
            tools=[*TASK_TOOLS, CREATE_SUBAGENTS, SEND_MESSAGE, WAIT_FOR_MESSAGE],
            dispatch={**task_handlers, "create_subagents": create_subagents},
            show_clock=show_clock,
        )
    finally:  # the answer is in; helpers still running are no longer needed
        for t in [progress, *helpers.values()]:
            t.cancel()
        await asyncio.gather(progress, *helpers.values(), return_exceptions=True)
    return {
        "answer": answer,
        "seconds": round(clock.elapsed),
        "helpers": len(helpers),
        "budget": budget,
        "tool_calls": total_calls(hub),
        "usage": {agent: dict(counts) for agent, counts in hub.usage.items()},
        "log": hub.log,
    }

## 端到端测试

在加入你自己的工具之前，先做一次简短运行来检查接线是否正确。`TASK_TOOLS` 为空时团队依靠模型自身的知识工作，而下面的问题要求主智能体使用它的子智能体，这样才有可看的内容。这是一次检查，不是示例任务。运行结束后，三个小型确定性读取器会打印本次运行日志记录到的内容：

- `print_summary` 打印答案以及每个智能体的工具使用次数。
- `show_first_prompts` 打印主智能体和第一个子智能体收到的第一条消息，以及紧随其后的
  时钟行。在带压力的分支中，那句“时间很重要”的话位于两者末尾。
- `show_clock_lines` 按智能体打印它被展示过的时钟行。子智能体的第一行是
  团队的已用时间，而不是 `0s`。
- `show_timeline` 打印主智能体何时派生子智能体、每个子智能体何时汇报，以及答案何时
  到来；如果有预算，还会对照预算展示。

要改为检查预算分支，换成那行带注释的代码即可。

In [8]:
QUESTION = (
    "Wiring check. Spawn one helper for each of these four sorting algorithms: quicksort, "
    "mergesort, heapsort and timsort. Have each report the average and worst-case time complexity and whether "
    "the sort is stable. Then put the four reports in one small table, and say on the first line "
    "which of the four Python's built-in sort uses."
)


def print_summary(result: dict) -> None:
    budget = result.get("budget")
    against = f" against a {round(budget)}s budget" if budget else ""
    print("=" * 88)
    print(
        f"ANSWER after {result['seconds']}s wall-clock{against}, "
        f"{result['helpers']} helper(s) spawned:\n{result['answer']}\n"
    )
    for agent, counts in sorted(result["usage"].items(), key=lambda kv: _agent_order(kv[0])):
        print(f"  {agent:<8} " + "  ".join(f"{tool} {n:>3}" for tool, n in sorted(counts.items())))
    print(f"  {'TOTAL':<8} task-tool calls {result['tool_calls']:>3}")


def _agent_order(agent: str) -> tuple:
    return (agent != "lead", int(re.sub(r"\D", "", agent) or 0))


def _tagged(result: dict, tag: str, agent: str | None = None) -> list[dict]:
    log_ = result["log"]
    return [e for e in log_ if e["text"].startswith(tag) and agent in (None, e["agent"])]


def show_first_prompts(result: dict, width: int = 200) -> None:
    """The first message the lead and the first helper received, and the clock line after it."""
    for agent in ("lead", "helper1"):
        tasks = _tagged(result, "[task] ", agent)
        if not tasks:
            continue
        text = tasks[0]["text"][len("[task] ") :]
        if len(text) > 2 * width:
            text = f"{text[:width]} [...] {text[-width:]}"
        clocks = _tagged(result, "[clock] ", agent)
        line = clocks[0]["text"][len("[clock] ") :] if clocks else "(no clock line)"
        print(f"{agent}, first message:\n  {text}\n  {line}\n")


def show_clock_lines(result: dict, limit: int = 8) -> None:
    """Per agent, the clock lines it was shown, in order."""
    seen = defaultdict(list)
    for e in _tagged(result, "[clock] "):
        seen[e["agent"]].append(e["text"][len("[clock] ") :])
    if not seen:
        print("No agent was shown a clock in this run.")
    for agent in sorted(seen, key=_agent_order):
        lines = seen[agent]
        more = f" -> ... ({len(lines) - limit} more)" if len(lines) > limit else ""
        print(f"{agent:<8} {' -> '.join(lines[:limit])}{more}")


def show_timeline(result: dict) -> None:
    """When the lead spawned helpers, when each reported, and when the answer came."""
    budget = result.get("budget")
    events = [(0, "lead starts" + (f", budget {round(budget)}s" if budget else ""))]
    for e in _tagged(result, "-> create_subagents(", "lead"):
        names = re.findall(r"helper\d+", e["text"].split("= spawned:")[-1])
        if names:
            events.append((e["seconds"], f"lead spawns {', '.join(names)}"))
    for e in _tagged(result, "[report] "):
        events.append((e["seconds"], f"{e['agent']} reports"))
    verdict = ""
    if budget:
        gap = round(budget) - result["seconds"]
        verdict = f" ({gap}s inside the budget)" if gap >= 0 else f" ({-gap}s past the budget)"
    events.append((result["seconds"], f"lead answers{verdict}"))
    for seconds, what in sorted(events, key=lambda x: x[0]):
        print(f"{seconds:>4}s  {what}")

In [9]:
# Smoke test: one pressured run, then read back what the agents were shown
result = await run_team(QUESTION, latency_pressure=True)
# result = await run_team(QUESTION, latency_budget_seconds=60)  # the budget arm instead
print_summary(result)
print("\n--- what the agents were told ---")
show_first_prompts(result)
print("--- the clock lines each agent saw ---")
show_clock_lines(result)
print("\n--- timeline ---")
show_timeline(result)

ANSWER after 9s wall-clock, 4 helper(s) spawned:
**Timsort** — Python's built-in `sorted()` / `list.sort()` uses it.

| Algorithm | Average | Worst | Stable |
|-----------|---------|-------|--------|
| Quicksort | O(n log n) | O(n²) | No |
| Mergesort | O(n log n) | O(n log n) | Yes |
| Heapsort  | O(n log n) | O(n log n) | No |
| Timsort   | O(n log n) | O(n log n) | Yes |

All four helpers reported back; results match standard references.

  TOTAL    task-tool calls   0

--- what the agents were told ---
lead, first message:
  Wiring check. Spawn one helper for each of these four sorting algorithms: quicksort, mergesort, heapsort and timsort. Have each report the average and worst-case time complexity and whether the sort i [...] mall table, and say on the first line which of the four Python's built-in sort uses. Time matters here: do not spend time that can be avoided, and the earlier a correct result is obtained, the better.
  [elapsed 0s]

helper1, first message:
  Report, in one 

## 结论

我们给“主智能体加子智能体”的团队配了一个共享时钟，并在每一轮之前展示给每个智能体。于是一句时延压力，或者时钟行上的一条预算，就是改变团队配速方式的东西。

- **选择压力**，适用于越快越好、而你心中没有具体数字的场景。它不需要任何
  调参。根据我们的经验，它对答案质量的影响很小，但本笔记本没有度量这一点，所以请在你自己的任务上验证。
- **选择预算**，适用于你有一个时延目标、并希望团队对照
  它自行配速的场景。
- **在你自己的任务上**，保留 `TeamClock`、`clock_line`、`stamp_elapsed` 和 `TIME_MATTERS`，
  并带上你自己的工具、系统提示词和问题。挑选周全与速度彼此拉扯的工作：没有可取舍之处的任务在每个分支看起来都一样。这里的替换点是 `TASK_TOOLS`、`make_task_handlers`、两个系统提示词以及你的问题。

预算更适合较长的任务，而不适合很短的任务。当整个任务只需一两分钟时，其中一半几乎容不下任何实际工作，Claude 更可能超出预算而不是放弃任务。请把这里的代码当作你自己试验时延激励的起点。

在采用更快的分支之前，请检查它的答案对你的任务是否仍然足够好。

顺理成章的下一步是度量：在你自己的任务上把每个分支各运行几次，比较墙钟时间。关于编排本身，参见 [异步多智能体编排](async_multi_agent_orchestration.ipynb)。